In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/train.csv")

print("Original Shape:", df.shape)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Strip spaces
for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()

# Replace missing strings
df.replace(
    ["NaN","nan","","NULL","null"],
    np.nan,
    inplace=True
)

# Numeric conversion
numeric_cols = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Vehicle_condition",
    "multiple_deliveries"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Delivery time
df["Time_taken(min)"] = (
    df["Time_taken(min)"]
    .astype(str)
    .str.extract("(\d+)")
    .astype(float)
)

# Date conversion
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    dayfirst=True,
    errors="coerce"
)

# Missing values
df["Delivery_person_Age"].fillna(
    df["Delivery_person_Age"].median(),
    inplace=True
)

df["Delivery_person_Ratings"].fillna(
    df["Delivery_person_Ratings"].median(),
    inplace=True
)

df["Road_traffic_density"].fillna(
    df["Road_traffic_density"].mode()[0],
    inplace=True
)

df["Weatherconditions"].fillna(
    df["Weatherconditions"].mode()[0],
    inplace=True
)

df["Festival"].fillna("No", inplace=True)

df["City"].fillna(
    df["City"].mode()[0],
    inplace=True
)

# Remove invalid ages
df = df[
    (df["Delivery_person_Age"] >= 18)
    &
    (df["Delivery_person_Age"] <= 70)
]

# Remove invalid ratings
df = df[
    (df["Delivery_person_Ratings"] >= 1)
    &
    (df["Delivery_person_Ratings"] <= 5)
]

df.reset_index(drop=True, inplace=True)

print("Final Shape:", df.shape)

df.to_csv(
    "../data/processed/cleaned_delivery.csv",
    index=False
)

print("Cleaning Completed")